# Gene Expression Prediction Model Preparation and Training Tutorial

This tutorial demonstrates how to prepare datasets and train a Gene Expression Prediction (GEP) model using the TMEformer framework. The GEP model is essential for downstream In Silico Perturbation (ISP) analysis.

## Overview

In this tutorial, you will learn how to:
1. Prepare fine-tuning dataset from cells expressing marker genes
2. Train the GEP model using the fine-tuning dataset
3. Prepare ISP perturbation dataset for downstream analysis

## Prerequisites

Before running this notebook, you need to:
- Have the `TMEformer` package installed
- Have pre-trained model checkpoints available
- Have processed single-cell data in the expected format

## Step 1: Prepare Fine-tuning Dataset

First, we prepare the fine-tuning dataset by selecting epithelial cells that express at least one of the marker genes. This ensures the model learns to predict expression patterns for relevant cell types.

In [2]:
import os
import json
import scanpy as sc
import pandas as pd
import numpy as np
from datasets import load_from_disk

from TMEformer.tme import TmeModeling_utils as tu
from TMEformer.tme import TmeModeling_utils_isp_gep as tu_isp_gep

/home/liss/miniconda3/envs/geneformer_v3/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Configure working directory and project
work_dir = "/dataSSD7T/liss/work/scPCa/model_paper/TMEformer-analysis/"
proj = "xenium"
proj_data_dir = work_dir + f"data/{proj}/"

# Load marker gene set
with open(work_dir + "isp_gene_exp/MODEL_FT_GENE_SET.json", "r") as f:
    FT_Multi_SETs = json.load(f)

set_id = "SET1"
marker_genes = FT_Multi_SETs[set_id]
print(f"Marker genes for {set_id}: {marker_genes}")

# Maximum number of cells to use
max_cells = 100_000

Marker genes for SET1: ['SYP', 'CHGA', 'ENO2', 'NCAM1']


In [4]:
# Load annotated data and metadata
adata = sc.read(proj_data_dir + f"/processed/{proj}_annotated.h5ad")
print(f"Loaded data shape: {adata.shape}")

obsmeta = pd.read_csv(proj_data_dir + f"/processed/{proj}_obsmeta.csv")
print(f"Loaded metadata with {len(obsmeta)} cells")

Loaded data shape: (1495698, 5051)
Loaded metadata with 1495698 cells


In [5]:
# Filter epithelial cells expressing at least one marker gene
gep_genes = pd.DataFrame(np.array(adata[:, marker_genes].X.todense()), columns=marker_genes)
obsmeta_gep = obsmeta.merge(gep_genes, left_index=True, right_index=True)
obsmeta_gep_epi = obsmeta_gep[obsmeta_gep["cell_type"] == "Epithelia"].copy()

# Count number of expressed marker genes per cell
obsmeta_gep_epi['num_labels'] = (obsmeta_gep_epi[marker_genes] > 0).sum(axis=1)
obsmeta_gep_epi_exp = obsmeta_gep_epi.query("num_labels > 0")

print(f"Epithelial cells expressing marker genes: {len(obsmeta_gep_epi_exp)}")
print("\nExpression counts per gene:")
print(obsmeta_gep_epi_exp[marker_genes].astype(bool).sum(axis=0))

Epithelial cells expressing marker genes: 158974

Expression counts per gene:
SYP       61680
CHGA     118739
ENO2      10278
NCAM1      8844
dtype: int64


In [6]:
# Balance cells expressing single marker genes
obsmeta_gep_epi_exp_L2 = obsmeta_gep_epi_exp.query("num_labels >= 2")
obsmeta_gep_epi_exp_L1 = obsmeta_gep_epi_exp.query("num_labels == 1")

# Sample to balance L1 cells
melt_exp_L1 = obsmeta_gep_epi_exp_L1[marker_genes].melt(
    var_name='marker', value_name='exp', ignore_index=False
).query("exp > 0")

min_freq = melt_exp_L1["marker"].value_counts().min()
obsmeta_gep_epi_exp_L1_sampled = obsmeta_gep_epi_exp_L1.loc[
    melt_exp_L1.groupby("marker").sample(min_freq, random_state=42).index, :
]

# Combine sampled L1 and all L2 cells
obsmeta_sub = pd.concat([obsmeta_gep_epi_exp_L1_sampled, obsmeta_gep_epi_exp_L2]).sort_index()

# Cap at max_cells if needed
if obsmeta_sub.shape[0] > max_cells:
    obsmeta_sub = obsmeta_sub.sample(n=max_cells, random_state=42).sort_index()

print(f"Final dataset size: {len(obsmeta_sub)} cells")
print("\nExpression counts after sampling:")
print(obsmeta_sub[marker_genes].astype(bool).sum(axis=0))

Final dataset size: 63785 cells

Expression counts after sampling:
SYP      41486
CHGA     44157
ENO2      9865
NCAM1     8844
dtype: int64


In [7]:
# Create gene expression dictionary
gep_dict = obsmeta_sub.set_index('cell_id')[marker_genes].to_dict(orient='index')
gep_dict = {k: list(v.values()) for k, v in gep_dict.items()}

# Load and select dataset
dataset = load_from_disk(work_dir + f"data/{proj}/datasets/{proj}.dataset")
dataset = dataset.select(obsmeta_sub["cell_id"].values - 1)

assert dataset["cell_id"] == obsmeta_sub['cell_id'].values.tolist()

# Convert marker genes to token IDs
gene_token_ids = [tu.symbol2token(gene) for gene in marker_genes]
print(f"Gene token IDs: {gene_token_ids}")

Gene token IDs: [2590, 2338, 3894, 9064]


In [8]:
# Create and save fine-tuning dataset
version = "v3"
dataset_path = f"data/{proj}/datasets_ft/{version}/{proj}_epi_gep_{set_id}.dataset"

if os.path.exists(work_dir + dataset_path):
    print(f"{dataset_path} exists, skip")
else:
    dataset_tme_v = load_from_disk(work_dir + f"data/{proj}/datasets/{proj}_TME_{version}.dataset")
    dataset_tme_v.cleanup_cache_files()
    dataset_tme_v = dataset_tme_v.select(np.array(dataset["cell_id"]) - 1)
    
    assert dataset_tme_v['cell_id'] == dataset['cell_id']
    
    # Process dataset: delete marker gene tokens and add gene expression
    dataset_tme_v = dataset_tme_v.map(
        tu.delete_multi_gene_token_and_add_gep, num_proc=10,
        fn_kwargs={"gene_token_ids": gene_token_ids, "gep_dict": gep_dict, "scale": False}
    )
    
    dataset_tme_v.save_to_disk(work_dir + dataset_path)
    print(f"Saved fine-tuning dataset to {dataset_path}")

data/xenium/datasets_ft/v3/xenium_epi_gep_SET1.dataset exists, skip


## Step 2: Train GEP Model

Now we train the GEP model using the fine-tuning dataset. This step uses the `tmeformer-gep-ft-model` command.

### Training Command

```bash
tmeformer-gep-ft-model --device 1 \
--proj xenium --model_id GF_D1120_06 --ft_model_id FT1120_06 \
--freeze_layers -6 --gene_set SET1 \
--train_mode cv_all --attr_col patch
```


In [ ]:
# Run model training (uncomment to execute)
# !tmeformer-gep-ft-model --device 1 \
# --proj xenium --model_id GF_D1120_06 --ft_model_id FT1120_06 \
# --freeze_layers -6 --gene_set SET1 \
# --train_mode cv_all --attr_col patch

## Step 3: Prepare ISP Perturbation Dataset

After training the GEP model, we prepare the ISP perturbation dataset. Unlike the fine-tuning dataset, this dataset randomly samples epithelial cells (not restricted to those expressing marker genes).

In [4]:
# Load metadata for ISP dataset
adata = sc.read(proj_data_dir + f"processed/{proj}_annotated.h5ad")
obsmeta = pd.read_csv(proj_data_dir + f"processed/{proj}_obsmeta.csv")

# Get valid marker genes
valid_marker_genes = [gene for gene in marker_genes if gene in adata.var_names]
print(f"Valid marker genes: {valid_marker_genes}")

# Annotate gene expression
if len(valid_marker_genes) > 0:
    gep_genes = pd.DataFrame(np.array(adata[:, valid_marker_genes].X.toarray()), columns=valid_marker_genes)
    obsmeta_gep = obsmeta.merge(gep_genes, left_index=True, right_index=True)
else:
    obsmeta_gep = obsmeta.copy()

# Fill missing genes with 0
for gene in list(set(marker_genes) - set(valid_marker_genes)):
    obsmeta_gep[gene] = 0.0

# Sample epithelial cells
obsmeta_gep_epi = obsmeta_gep[obsmeta_gep["cell_type"] == "Epithelia"].copy()
n_cells = len(load_from_disk(work_dir + f"isp_gene_exp/{proj}/datasets/ft_v3_{set_id}_labeled.dataset"))
obsmeta_sub = obsmeta_gep_epi.sample(n=n_cells, random_state=42)

print(f"Sampled {len(obsmeta_sub)} cells for ISP dataset")

Valid marker genes: ['SYP', 'CHGA', 'ENO2', 'NCAM1']
Sampled 63785 cells for ISP dataset


In [5]:
# Create gene expression dictionary
gep_dict = obsmeta_sub.set_index('cell_id')[marker_genes].to_dict(orient='index')
gep_dict = {k: list(v.values()) for k, v in gep_dict.items()}

# Load and select dataset
dataset = load_from_disk(proj_data_dir + f"/datasets/{proj}.dataset")
dataset = dataset.select(obsmeta_sub["cell_id"].values - 1)

assert dataset["cell_id"] == obsmeta_sub['cell_id'].values.tolist()

# Convert marker genes to token IDs
gene_token_ids = [tu.symbol2token(gene) for gene in marker_genes]
gene_token_ids

[2590, 2338, 3894, 9064]

In [6]:
# Create and save ISP dataset
version = "v3"
dataset_path = work_dir + f"isp_gene_exp/{proj}/datasets/isp_{version}_{set_id}.dataset/"

if os.path.exists(dataset_path):
    print(f"{dataset_path} already exists.")
else:
    dataset_tme_v = load_from_disk(proj_data_dir + f"/datasets/{proj}_TME_{version}.dataset")
    dataset_tme_v.cleanup_cache_files()
    
    id_to_idx = {cid: i for i, cid in enumerate(dataset_tme_v["cell_id"])}
    indices = [id_to_idx[cid] for cid in dataset["cell_id"] if cid in id_to_idx]
    dataset_tme_v = dataset_tme_v.select(indices)
    
    assert dataset_tme_v['cell_id'] == dataset['cell_id']
    
    # Process dataset
    dataset_tme_v = dataset_tme_v.map(
        tu.delete_multi_gene_token_and_add_gep, num_proc=10,
        fn_kwargs={"gene_token_ids": gene_token_ids, "gep_dict": gep_dict}
    )
    dataset_tme_v = dataset_tme_v.rename_column("geps", "label")
    dataset_tme_v.save_to_disk(dataset_path)
    print(f"Saved ISP dataset to {dataset_path}")

Saved ISP dataset to /dataSSD7T/liss/work/scPCa/model_paper/TMEformer-analysis/isp_gene_exp/xenium/datasets/isp_v3_SET1.dataset/


## Step 4: Prepare Background Gene Lists

Finally, we prepare background gene lists for random gene perturbation analysis.

In [9]:
# Prepare background gene lists
tu_isp_gep.prep_bg_gene_lists(
    proj, "SET1", 1, pred_cells="internal", work_dir=work_dir, existed=True
)


/dataSSD7T/liss/work/scPCa/model_paper/TMEformer-analysis/isp_gene_exp/xenium/datasets/background/random_1genes_bg_cells_Model_SET1.dict already exists
